# Deblurring tables for paper

In [3]:
# ============================================================
# DEBLURRING RESULTS TABLE (PSNR + SSIM)
# Few-shot = trlim5, Many-shot = trlim15
# Averaged across cvfold0/1/2 within each datatype+trlim
# Exports CSV + LaTeX with best-per-column bolded
# ============================================================

from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import re
import numpy as np
import pandas as pd

# ---------------------------
# CONFIG
# ---------------------------
OUTDIR = Path("/midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables")
OUTDIR.mkdir(parents=True, exist_ok=True)

FOLDS = [0, 1, 2]
TRLIMS = [5, 15]  # few-shot, many-shot
DECIMALS = 2

DTYPES_GT = ["amyloid_plaque_patches", "cell_nucleus_patches", "vessels_patches"]
DTYPE_CANON = {
    "amyloid_plaque_patches": "Amyloid Plaque",
    "cell_nucleus_patches": "Cell Nucleus",
    "vessels_patches": "Vessels",
}

# roots
DEBLUR_ROOTS: Dict[str, Path] = {
    "UNet Img+Txt": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_deblur_preds_super_sweep2_v2"),
    "UNet Img":     Path("/midtier/paetzollab/scratch/ads4015/temp_selma_deblur_preds_bright_sweep_26"),
    "UNet Rand":    Path("/midtier/paetzollab/scratch/ads4015/temp_selma_deblur_preds_unet_random2_v2"),
    "SwinUNETR Img+Txt": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_deblur_preds_autumn_sweep_27_v2_subset"),
    "SwinUNETR Img":     Path("/midtier/paetzollab/scratch/ads4015/temp_selma_deblur_preds_expert_sweep_31_v2_subset"),
    "SwinUNETR Rand":    Path("/midtier/paetzollab/scratch/ads4015/temp_selma_deblur_preds_random_v2_subset"),
}

MODEL_ORDER = [
    "UNet Img+Txt",
    "SwinUNETR Img+Txt",
    "UNet Img",
    "SwinUNETR Img",
    "UNet Rand",
    "SwinUNETR Rand",
]

# ---------------------------
# HELPERS
# ---------------------------

def safe_mean(xs: List[float]) -> float:
    xs = [x for x in xs if x is not None and np.isfinite(x)]
    return float(np.mean(xs)) if xs else float("nan")

def pick_run_dir_for_fold_trlim(dtype_dir: Path, fold: int, trlim: int) -> Optional[Path]:
    """
    Under:
      <root>/preds/<dtype>/
    choose most recent run directory matching cvfold{fold}_trl(?:im|m){trlim}.
    Example:
      cvfold0_trlim5_fttr4_ftval1_tst2_..._seed100
    """
    if not dtype_dir.exists():
        return None
    pat = re.compile(rf"cvfold{fold}_trl(?:im|m){trlim}(?:_|$)") # unfortunately, some folders have a type (trlm instead of trlim)
    run_dirs = [d for d in dtype_dir.iterdir() if d.is_dir() and pat.search(d.name)]
    if not run_dirs:
        return None
    run_dirs.sort(key=lambda d: d.stat().st_mtime, reverse=True)
    return run_dirs[0]

def read_metrics_test_csv(run_dir: Path) -> Optional[pd.DataFrame]:
    """
    metrics_test.csv is typically in:
      <run_dir>/preds/metrics_test.csv
    but we search robustly under run_dir for any metrics_test.csv.
    """
    if run_dir is None or not run_dir.exists():
        return None

    c1 = run_dir / "preds" / "metrics_test.csv"
    if c1.exists():
        return pd.read_csv(c1)

    hits = list(run_dir.rglob("metrics_test.csv"))
    if not hits:
        return None
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return pd.read_csv(hits[0])

def eval_fold_trlim(root: Path, dtype_gt: str, fold: int, trlim: int) -> Tuple[float, float]:
    """
    Returns (mean_psnr, mean_ssim) for one fold+trlim, averaged over rows in metrics_test.csv.
    """
    dtype_dir = root / "preds" / dtype_gt
    run_dir = pick_run_dir_for_fold_trlim(dtype_dir, fold=fold, trlim=trlim)
    dfm = read_metrics_test_csv(run_dir)
    if dfm is None or dfm.empty:
        return (float("nan"), float("nan"))

    psnr = pd.to_numeric(dfm.get("psnr", pd.Series(dtype=float)), errors="coerce")
    ssim = pd.to_numeric(dfm.get("ssim", pd.Series(dtype=float)), errors="coerce")
    return (float(psnr.mean(skipna=True)), float(ssim.mean(skipna=True)))

def eval_model_dtype_trlim(root: Path, dtype_gt: str, trlim: int) -> Dict[str, float]:
    """
    Average across cvfold0/1/2 for a given datatype + trlim.
    """
    psnrs, ssims = [], []
    for fold in FOLDS:
        p, s = eval_fold_trlim(root, dtype_gt, fold=fold, trlim=trlim)
        psnrs.append(p)
        ssims.append(s)
    return {"PSNR": safe_mean(psnrs), "SSIM": safe_mean(ssims)}

# LaTeX helpers: bold max per numeric column
def latex_bold_max_per_column(df_num: pd.DataFrame, decimals: int = 2, na_rep: str = "—") -> pd.DataFrame:
    fmt = f"{{:.{decimals}f}}"
    out = pd.DataFrame(index=df_num.index, columns=df_num.columns, dtype=object)

    for col in df_num.columns:
        s = pd.to_numeric(df_num[col], errors="coerce")
        maxv = s.max(skipna=True)
        col_out = []
        for v in s.values:
            if not np.isfinite(v):
                col_out.append(na_rep)
            else:
                txt = fmt.format(float(v))
                if np.isfinite(maxv) and np.isclose(v, maxv, rtol=0, atol=1e-12):
                    txt = rf"\textbf{{{txt}}}"
                col_out.append(txt)
        out[col] = col_out
    return out

def latex_tabular_colspec(n_numeric_cols: int) -> str:
    return "@{}" + "l" + ("c" * n_numeric_cols) + "@{}"

def replace_first_tabular_colspec(latex_str: str, colspec: str) -> str:
    return re.sub(
        r"\\begin\{tabular\}\{[^}]*\}",
        rf"\\begin{{tabular}}{{{colspec}}}",
        latex_str,
        count=1,
    )

def inject_table_formatting(
    latex_str: str,
    add_centering: bool = True,
    fontsize_cmd: str = r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt: int = 2,
    arraystretch: float = 1.05,
) -> str:
    lines = latex_str.splitlines()
    out = []
    injected = False
    for line in lines:
        out.append(line)
        s = line.strip()
        if (not injected) and (s.startswith(r"\begin{table}") or s.startswith(r"\begin{sidewaystable}")):
            if add_centering:
                out.append(r"\centering")
            out.append(fontsize_cmd)
            out.append(rf"\setlength{{\tabcolsep}}{{{tabcolsep_pt}pt}}")
            out.append(rf"\renewcommand{{\arraystretch}}{{{arraystretch}}}")
            injected = True
    return "\n".join(out)

# ---------------------------
# BUILD WIDE TABLE: (Datatype, Shot, Metric)
# ---------------------------
SHOT_LABEL = {5: "Few-shot", 15: "Many-shot"}

records = []
for model in MODEL_ORDER:
    root = DEBLUR_ROOTS[model]
    row = {"Model": model}
    for dtype_gt in DTYPES_GT:
        dtype_name = DTYPE_CANON[dtype_gt]
        for trlim in TRLIMS:
            shot = SHOT_LABEL[trlim]
            m = eval_model_dtype_trlim(root, dtype_gt, trlim=trlim)
            row[(dtype_name, shot, "PSNR")] = m["PSNR"]
            row[(dtype_name, shot, "SSIM")] = m["SSIM"]
    records.append(row)

df_deblur = pd.DataFrame.from_records(records).set_index("Model")
df_deblur.columns = pd.MultiIndex.from_tuples(df_deblur.columns)

# Save numeric CSV
csv_out = OUTDIR / "deblur_results.csv"
df_deblur.round(6).to_csv(csv_out)
print(f"[Saved] {csv_out}")

# ---------------------------
# LaTeX: bold best per column
# ---------------------------
df_num = df_deblur.round(DECIMALS)
df_tex = latex_bold_max_per_column(df_num, decimals=DECIMALS, na_rep="—")

latex_str = df_tex.to_latex(
    escape=False,
    multicolumn=True,
    multirow=True,
    caption=(
        "Deblurring performance (PSNR and SSIM) for few-shot (trlim5) and many-shot (trlim15). "
        "Values are averaged across 3 CV folds."
    ),
    label="tab:deblur_results",
    bold_rows=False,
    longtable=False,
    index=True,
)

# Center datatype headers (each datatype spans 4 cols: Few(PSNR,SSIM) + Many(PSNR,SSIM))
for pretty in DTYPE_CANON.values():
    latex_str = latex_str.replace(rf"\multicolumn{{4}}{{r}}{{{pretty}}}", rf"\multicolumn{{4}}{{c}}{{{pretty}}}")

# Center shot headers across PSNR/SSIM
latex_str = latex_str.replace(r"\multicolumn{2}{r}{Few-shot}",  r"\multicolumn{2}{c}{Few-shot}")
latex_str = latex_str.replace(r"\multicolumn{2}{r}{Many-shot}", r"\multicolumn{2}{c}{Many-shot}")

# Tight colspec: Model + (3 datatypes * 4 cols) = 12 numeric cols
latex_str = replace_first_tabular_colspec(latex_str, latex_tabular_colspec(n_numeric_cols=12))

# Inject compact formatting
latex_str = inject_table_formatting(
    latex_str,
    add_centering=True,
    fontsize_cmd=r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt=2,
    arraystretch=1.05,
)

tex_out = OUTDIR / "deblur_results.tex"
tex_out.write_text(latex_str)
print(f"[Saved] {tex_out}")

# Optional display
display(df_deblur.round(DECIMALS))


[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/deblur_results.csv
[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/deblur_results.tex


Amyloid Plaque                       Cell Nucleus        \
                        Few-shot       Many-shot           Few-shot         
                            PSNR  SSIM      PSNR  SSIM         PSNR  SSIM   
Model                                                                       
UNet Img+Txt               25.25  0.65     26.42  0.69        17.76  0.80   
SwinUNETR Img+Txt          23.69  0.58     24.97  0.64        22.71  0.79   
UNet Img                   25.81  0.66     26.66  0.69        23.80  0.88   
SwinUNETR Img              20.93  0.57     24.89  0.65        22.34  0.81   
UNet Rand                  24.75  0.63     26.13  0.67        20.01  0.81   
SwinUNETR Rand             23.84  0.61     25.51  0.66        22.45  0.79   

                                   Vessels                        
                  Many-shot       Few-shot       Many-shot        
                       PSNR  SSIM     PSNR  SSIM      PSNR  SSIM  
Model                                                             
UNet Img+Txt          24.36  0.89    23.60  0.78     29.32  0.90  
SwinUNETR Img+Txt     24.03  0.85    26.28  0.85     30.89  0.90  
UNet Img              24.82  0.89    27.16  0.88     30.38  0.92  
SwinUNETR Img         23.78  0.86    21.26  0.73     19.68  0.69  
UNet Rand             23.02  0.87    27.95  0.88     29.30  0.90  
SwinUNETR Rand        24.33  0.85    26.03  0.80     30.34  0.90